In [ ]:
import json

objectdict = json.load(open("objects-raw.json", "r"))

for id in objectdict.keys():
    valuesdictarray = objectdict[id]
    valuesdict = dict()
    for i in range(len(valuesdictarray)):
        valuesdict |= valuesdictarray[i]
    valuesdict.__delitem__("Provenance")
    objectdict[id] = valuesdict

json.dump(objectdict, open("objects.json", "w"), indent=2)

In [ ]:
import json
from contextlib import redirect_stdout

objects = json.load(open("objects.json", "r"))

tableheader = r"""
\begin{table}[t]
    \setlength{\tabcolsep}{2pt}
    \footnotesize
    \begin{tabularx}{\linewidth}{lccX}
        \hline
        Name&Position&Type&Other Names\\
        \hline
"""

tablefooter = r"""
        \hline
    \end{tabularx}
\end{table}
\clearpage
"""

nlines = 50

def maketable(thiscatalog, ifirst, ilast):
    with open("%s-table.tex" % thiscatalog, "w") as f:
        iline = 0
        with redirect_stdout(f):
            for i in range(ifirst, ilast + 1):
                if iline % nlines == 0:
                    print(tableheader)
                for id in objects:
                    object = objects[id]
                    if "Catalog %s" % thiscatalog not in object:
                        continue
                    if object["Catalog %s" % thiscatalog] != i:
                        continue
                    thisname = "%s%d" % (thiscatalog, object["Catalog %s" % thiscatalog])
                    othercatalogs = ["NGC", "IC", "M", "C", "U", "Mel", "Cr"]
                    othernames = []
                    for othercatalog in othercatalogs:
                        if (
                            thiscatalog != othercatalog
                            and "Catalog %s" % othercatalog in object
                        ):
                            if othercatalog in ["M", "C", "U"]:
                                othernames.append(
                                    "%s%s"
                                    % (othercatalog, object["Catalog %s" % othercatalog])
                                )
                            else:
                                othernames.append(
                                    "%s %s"
                                    % (othercatalog, object["Catalog %s" % othercatalog])
                                )
                    if "Name" in object:
                        othernames.append(object["Name"])
                    othernames = " = ".join(othernames)
                    line = "%-4s &$%04.1f$ $%+03.0f$&%-3s&%s\\\\" % (
                        thisname,
                        object["alpha"] / 15,
                        object["delta"],
                        object["Type"],
                        othernames,
                    )
                    print(line, file=sys.stderr)
                    print(line)
                    iline += 1
                    if iline % nlines == 0:
                        print(tablefooter)
            if iline % nlines != 0:
                print(tablefooter)

maketable("M", 1, 110)
maketable("C", 1, 109)
maketable("U", 1, 87)